In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Configurer le WebDriver pour Chrome
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# URL cible
url = "https://agences.aramisauto.com/fr?_gl=1*3b2wlm*_gcl_au*MTI4ODU3NjQ3NS4xNzMxODYyNDM2*_ga*MTU0MDM3MDkwOS4xNzMxODYyNDM4*_ga_V9MLYR8MMH*MTczMTg2MjQzNy4xLjEuMTczMTg2MjQ0MC4wLjAuMA.."
driver.get(url)

# Attendre que la section StoreListSlider soit visible
try:
    store_list_section = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, 'StoreListSlider'))
    )
except Exception as e:
    print(f"La section 'StoreListSlider' n'a pas été trouvée : {e}")
    driver.quit()
    exit()

# Scroller pour charger tous les garages
last_height = driver.execute_script("return arguments[0].scrollHeight", store_list_section)
while True:
    driver.execute_script("arguments[0].scrollTo(0, arguments[0].scrollHeight);", store_list_section)
    time.sleep(2)  # Attendre que le contenu se charge

    new_height = driver.execute_script("return arguments[0].scrollHeight", store_list_section)
    if new_height == last_height:
        break
    last_height = new_height

# Extraire les données
garage_titles = []
garage_addresses = []

# Trouver les éléments de garage
garages = driver.find_elements(By.CLASS_NAME, 'ItemMagasin')
if not garages:
    print("Aucun garage trouvé avec la classe 'ItemMagasin'.")
else:
    for garage in garages:
        # Récupérer le titre du garage
        try:
            title = garage.find_element(By.CSS_SELECTOR, '[itemprop="name"]').text
        except:
            title = "N/A"

        # Récupérer l'adresse complète
        try:
            address_element = garage.find_element(By.CLASS_NAME, 'adress-content')
            street_address = address_element.find_element(By.CSS_SELECTOR, '[itemprop="streetAddress"]').text
            postal_code = address_element.find_element(By.CSS_SELECTOR, '[itemprop="postalCode"]').text
            locality = address_element.find_element(By.CSS_SELECTOR, '[itemprop="addressLocality"]').text
            full_address = f"{street_address}, {postal_code} {locality}"
        except:
            full_address = "N/A"

        # Ajouter les informations aux listes
        garage_titles.append(title)
        garage_addresses.append(full_address)

# Fermer le navigateur
driver.quit()

# Vérifier si les données ont été collectées
if garage_titles and garage_addresses:
    # Créer un DataFrame avec les données extraites
    df = pd.DataFrame({
        'Title': garage_titles,
        'Address': garage_addresses
    })

    # Exporter les données dans un fichier Excel
    output_path = 'C:/Users/hugoj/Desktop/garages_aramisauto.xlsx'
    df.to_excel(output_path, index=False)
    print(f"Données extraites et exportées dans '{output_path}'")
else:
    print("Aucune donnée n'a été extraite.")



Données extraites et exportées dans 'C:/Users/hugoj/Desktop/garages_aramisauto.xlsx'
